# Reproducible Search Workflow for the Supervised Candidates

This notebook is a reusable search harness for the project's Linear Regression, Decision Tree and Random Forest candidates, plus a regularized-linear alternative. It centralizes the shared data contract, chronological split, expanding-window folds, MAE-first scoring, model specifications, search budgets and result schema. It does **not** rebuild the modeling matrix, rerun K-Means, replace the detailed candidate analyses, select a new final model or export production artifacts.

The source of truth is the merged as-of modeling matrix and its companion feature contract produced by notebook 13. That matrix already incorporates the active-operator population derived from the notebook 11/12 operator-by-municipality clustering method using only information available before each modeled period. Annual clustering artifacts are not loaded here because they would not be valid forecasting inputs.


## How to use this notebook

1. Run the contract and temporal-split cells first. They discover periods and dataset-dependent sizes from the persisted matrix.
2. Review `MODEL_SPECS`. Model-specific feature handling and parameter spaces live there; shared execution code does not contain model-specific branches beyond the declared strategy.
3. Set `FULL_SEARCH_MODELS` to one model name or `list(MODEL_SPECS)` and run the full-search cell. Leaving it as `[]` is the harness's idle default and skips the expensive search; this checked-in run sets it to `list(MODEL_SPECS)` to exercise every registered model specification.
4. Use the common result table and the validation leaderboard to compare candidates by mean validation MAE. Secondary metrics are diagnostics only.
5. Enable the holdout gate only after the team has selected a model and configuration from validation results. Holdout performance must never feed back into search or selection.

The optional smoke test uses one deliberately reduced configuration per candidate. Its outputs are structural checks, are labeled non-final, and must not be persisted or reported as selected-model evidence.


In [1]:
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any
import json
import os
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import (
    GridSearchCV, ParameterGrid, RandomizedSearchCV, cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.tree import DecisionTreeRegressor

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
print("scikit-learn version:", sklearn.__version__)
sklearn_major_minor = tuple(int(part) for part in sklearn.__version__.split(".")[:2])
if sklearn_major_minor < (1, 4):
    raise RuntimeError(
        "This workflow requires scikit-learn >= 1.4 because the tree candidates preserve "
        "the existing notebooks' native missing-value handling instead of inventing imputation."
    )

scikit-learn version: 1.9.0


## 1. Shared configuration and persisted inputs

The last `HOLDOUT_PERIODS` available periods form the untouched holdout. The immediately preceding `N_VALIDATION_FOLDS` periods are validation endpoints; each fold trains on every strictly earlier training period. This reproduces the project's forecasting design without hardcoding calendar months, row counts, operator IDs, cluster labels, scores, or selected hyperparameters from a previous artifact version. Default shuffled `KFold` is inappropriate because lag features contain realized earlier targets: random folds could train on a later row whose lag contains the target of a row being validated.


In [2]:
RANDOM_SEED = 42
HOLDOUT_PERIODS = 3
N_VALIDATION_FOLDS = 3
PRIMARY_METRIC = "mae"
SEARCH_N_JOBS = 1  # avoids nested parallelism; Random Forest parallelizes its own trees

UNIT_KEYS = ["ID_PLANO", "treatment_line_primary", "UF_PRESTADOR", "MES"]
TEMPORAL_COLUMN = "MES"

WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELING_ARTIFACT = PROCESSED_DIR / "north_region_mama_2025_modeling_matrix_atuantes_v1.parquet"
FEATURE_CONTRACT_ARTIFACT = PROCESSED_DIR / "north_region_mama_2025_modeling_matrix_atuantes_v1_feature_contract.csv"

for required_path in (MODELING_ARTIFACT, FEATURE_CONTRACT_ARTIFACT):
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Required merged artifact is missing: {required_path}. "
            "Run or restore notebook 13's persisted outputs; do not rebuild them in this notebook."
        )

modeling_matrix = pd.read_parquet(MODELING_ARTIFACT)
feature_contract = pd.read_csv(FEATURE_CONTRACT_ARTIFACT)
print("Modeling artifact:", MODELING_ARTIFACT.relative_to(REPO_ROOT))
print("Feature contract:", FEATURE_CONTRACT_ARTIFACT.relative_to(REPO_ROOT))

Modeling artifact: data/processed/north_region_mama_2025_modeling_matrix_atuantes_v1.parquet
Feature contract: data/processed/north_region_mama_2025_modeling_matrix_atuantes_v1_feature_contract.csv


## 2. Validate and resolve the modeling contract

The target and predictors are resolved from the companion contract rather than restated independently. Assertions stop on an empty artifact, schema drift, missing or duplicate observation keys, incompatible types, non-monthly timestamps, non-finite values, a negative target, or a mismatch between the persisted column order and the contract. Missing predictor values are allowed because lag and rolling features have expected burn-in gaps; their treatment belongs inside each model pipeline.


In [3]:
def validate_and_resolve_contract(matrix: pd.DataFrame, contract: pd.DataFrame):
    if matrix.empty:
        raise ValueError("The merged modeling matrix is empty. Update notebook 13 before running searches.")

    contract_columns = {"FEATURE_NAME", "ROLE"}
    missing_contract_columns = sorted(contract_columns - set(contract.columns))
    if missing_contract_columns:
        raise ValueError(f"Feature contract is missing columns: {missing_contract_columns}")

    target_rows = contract.loc[contract["ROLE"].eq("TARGET"), "FEATURE_NAME"].tolist()
    predictors = contract.loc[contract["ROLE"].eq("PREDICTOR"), "FEATURE_NAME"].tolist()
    if len(target_rows) != 1:
        raise ValueError(f"Expected exactly one TARGET in the feature contract; found {target_rows}.")
    if not predictors or len(predictors) != len(set(predictors)):
        raise ValueError("The PREDICTOR rows in the feature contract are empty or duplicated.")

    target = target_rows[0]
    expected_columns = UNIT_KEYS + [target] + predictors
    missing_columns = sorted(set(expected_columns) - set(matrix.columns))
    unexpected_columns = sorted(set(matrix.columns) - set(expected_columns))
    if missing_columns or unexpected_columns or list(matrix.columns) != expected_columns:
        raise ValueError(
            "The modeling artifact is incompatible with notebook 13's persisted contract. "
            f"Missing={missing_columns}; unexpected={unexpected_columns}; "
            "update the artifact and contract together before tuning."
        )

    if matrix[UNIT_KEYS].isna().any().any():
        raise ValueError("Observation keys contain missing values.")
    duplicate_count = int(matrix.duplicated(UNIT_KEYS).sum())
    if duplicate_count:
        raise ValueError(f"Found {duplicate_count} duplicate observation keys: {UNIT_KEYS}.")

    parsed_time = pd.to_datetime(matrix[TEMPORAL_COLUMN], errors="coerce")
    if parsed_time.isna().any():
        raise TypeError(f"{TEMPORAL_COLUMN} must contain valid monthly timestamps.")
    if not parsed_time.dt.is_month_start.all():
        raise ValueError(f"{TEMPORAL_COLUMN} must use month-start timestamps.")
    matrix = matrix.copy()
    matrix[TEMPORAL_COLUMN] = parsed_time

    numeric_columns = [target] + predictors
    non_numeric = [column for column in numeric_columns if not pd.api.types.is_numeric_dtype(matrix[column])]
    if non_numeric:
        raise TypeError(f"Target and predictors must be numeric; incompatible columns: {non_numeric}.")
    if matrix[target].isna().any() or not np.isfinite(matrix[target].to_numpy(dtype=float)).all():
        raise ValueError(f"{target} must be complete and finite.")
    if matrix[target].lt(0).any():
        raise ValueError(f"{target} must be non-negative.")
    predictor_values = matrix[predictors].to_numpy(dtype=float)
    if np.isinf(predictor_values).any():
        raise ValueError("Predictors contain infinite values; NaN burn-in values are allowed, infinity is not.")
    if "MONTH" in predictors and not matrix["MONTH"].eq(matrix[TEMPORAL_COLUMN].dt.month).all():
        raise ValueError("MONTH is inconsistent with MES; update notebook 13's artifact and feature contract together.")

    return matrix, target, predictors


modeling_matrix, TARGET_COLUMN, APPROVED_FEATURES = validate_and_resolve_contract(
    modeling_matrix, feature_contract
)
print("Target:", TARGET_COLUMN)
print("Approved predictors:", APPROVED_FEATURES)
print("Rows and periods are discovered at runtime:", len(modeling_matrix), modeling_matrix[TEMPORAL_COLUMN].nunique())


Target: volume_demanda_proxy
Approved predictors: ['DEMAND_LAG_1M', 'DEMAND_LAG_3M', 'DEMAND_LAG_6M', 'DEMAND_MEAN_3M', 'DEMAND_MEAN_6M', 'DEMAND_STD_3M', 'PORTE_ORDINAL', 'MONTH']
Rows and periods are discovered at runtime: 11526 11


## 3. Chronological holdout and expanding-window validation

The holdout is separated before any search object is instantiated. Validation folds refer only to row positions inside the training partition. Every fold validates one period and fits on all strictly earlier periods. The assertions below also prove that no holdout row ID occurs in a fold and that training, validation, and holdout periods do not overlap.


In [4]:
def build_temporal_partitions(matrix: pd.DataFrame):
    periods = pd.DatetimeIndex(sorted(matrix[TEMPORAL_COLUMN].unique()))
    minimum_periods = HOLDOUT_PERIODS + N_VALIDATION_FOLDS + 1
    if len(periods) < minimum_periods:
        raise ValueError(
            f"Need at least {minimum_periods} periods for an initial window, "
            f"{N_VALIDATION_FOLDS} validation folds, and {HOLDOUT_PERIODS} holdout periods; found {len(periods)}."
        )

    expected_periods = pd.date_range(periods.min(), periods.max(), freq="MS")
    if not periods.equals(expected_periods):
        missing_periods = expected_periods.difference(periods).strftime("%Y-%m").tolist()
        raise ValueError(f"The monthly sequence is not contiguous; missing periods: {missing_periods}.")

    holdout_period_index = periods[-HOLDOUT_PERIODS:]
    training_period_index = periods[:-HOLDOUT_PERIODS]
    validation_period_index = training_period_index[-N_VALIDATION_FOLDS:]
    holdout_start = holdout_period_index.min()

    ordered = matrix.sort_values(UNIT_KEYS).reset_index(drop=False).rename(columns={"index": "__source_row_id"})
    training = ordered.loc[ordered[TEMPORAL_COLUMN] < holdout_start].reset_index(drop=True)
    holdout = ordered.loc[ordered[TEMPORAL_COLUMN] >= holdout_start].reset_index(drop=True)
    if training.empty or holdout.empty:
        raise ValueError("Chronological partition produced an empty training or holdout set.")
    if not training[TEMPORAL_COLUMN].max() < holdout[TEMPORAL_COLUMN].min():
        raise AssertionError("Training periods must be strictly earlier than holdout periods.")
    if not set(training[TEMPORAL_COLUMN]).isdisjoint(set(holdout[TEMPORAL_COLUMN])):
        raise AssertionError("Training and holdout periods overlap.")

    folds = []
    holdout_row_ids = set(holdout["__source_row_id"])
    for validation_period in validation_period_index:
        fit_positions = np.flatnonzero(training[TEMPORAL_COLUMN].lt(validation_period))
        validation_positions = np.flatnonzero(training[TEMPORAL_COLUMN].eq(validation_period))
        if not len(fit_positions) or not len(validation_positions):
            raise ValueError(f"Empty fit or validation window for {validation_period:%Y-%m}.")
        fit_periods = training.iloc[fit_positions][TEMPORAL_COLUMN]
        validation_periods = training.iloc[validation_positions][TEMPORAL_COLUMN]
        if not fit_periods.max() < validation_periods.min():
            raise AssertionError("A fold contains training periods that are not strictly earlier than validation.")
        fold_row_ids = set(training.iloc[np.concatenate([fit_positions, validation_positions])]["__source_row_id"])
        if fold_row_ids & holdout_row_ids:
            raise AssertionError("Holdout rows appear in cross-validation folds.")
        folds.append((fit_positions, validation_positions))

    return training, holdout, folds, validation_period_index, holdout_period_index


training_data, holdout_data, TEMPORAL_FOLDS, validation_periods, holdout_periods = build_temporal_partitions(
    modeling_matrix
)
print("Training periods:", training_data[TEMPORAL_COLUMN].min().date(), "to", training_data[TEMPORAL_COLUMN].max().date())
print("Validation endpoints:", [period.strftime("%Y-%m") for period in validation_periods])
print("Untouched holdout periods:", [period.strftime("%Y-%m") for period in holdout_periods])
print("Fold sizes:", [(len(fit_rows), len(validation_rows)) for fit_rows, validation_rows in TEMPORAL_FOLDS])


Training periods: 2025-02-01 to 2025-09-01
Validation endpoints: ['2025-07', '2025-08', '2025-09']
Untouched holdout periods: ['2025-10', '2025-11', '2025-12']
Fold sizes: [(4238, 1086), (5324, 1154), (6478, 1238)]


### 3.1 Rolling-origin audit

Each validation fold is one **forecast origin**: the model is fit on every month strictly before the origin and scored on the origin month, and the origin rolls forward one month per fold. The assertions in Section 3 already prove that no fold sees a later month or a holdout row. This audit checks the remaining properties a rolling-origin design needs, fold by fold:

- **Consecutive origins and expanding windows.** The origins are consecutive months, every fit window starts at the first training month, and each window contains the previous one plus exactly one more month.
- **History burn-in is the only kind of missing predictor.** A predictor that is entirely missing in a fit window cannot be learned from that window. This is expected for the long lags at the earliest origin: the as-of matrix starts in February 2025, and a six-month lag has no source month before July. The audit lists every such predictor, requires it to be a history feature, and requires it to be observed in the origin month. Anything else would mean the design, not the burn-in, is starving a fold.
- **Reference behaviour.** The seasonal-naive forecast (`DEMAND_LAG_1M`, 0 where absent) is scored on each origin. The values must reproduce notebook 15's baseline table (cell 18: 37.471 / 28.743 / 26.728, mean 30.980), which shows that notebook 18's folds are the same origins the candidate notebooks tuned on.

In [5]:
HISTORY_FEATURE_PREFIX = "DEMAND_"
# Seasonal-naive MAE per origin, from notebook 15's committed baseline table (cell 18). These are
# reference values for a consistency check only; they select nothing.
NOTEBOOK_14_NAIVE_MAE_BY_ORIGIN = {"2025-07": 37.471, "2025-08": 28.743, "2025-09": 26.728}
NAIVE_REFERENCE_TOLERANCE = 5e-4  # the reference is printed to three decimals


def audit_rolling_origins(training, folds, origins):
    first_training_period = training[TEMPORAL_COLUMN].min()
    expected_origins = pd.date_range(origins[0], periods=len(origins), freq="MS")
    if not pd.DatetimeIndex(origins).equals(expected_origins):
        raise AssertionError("Validation origins are not consecutive months.")

    rows, previous_fit = [], None
    for (fit_positions, validation_positions), origin in zip(folds, origins):
        fit = training.iloc[fit_positions]
        validation = training.iloc[validation_positions]
        if fit[TEMPORAL_COLUMN].min() != first_training_period:
            raise AssertionError(f"Origin {origin:%Y-%m}: the fit window does not start at the first training month.")
        if previous_fit is not None:
            added_periods = set(fit[TEMPORAL_COLUMN]) - set(training.iloc[previous_fit][TEMPORAL_COLUMN])
            if not set(previous_fit) <= set(fit_positions) or len(added_periods) != 1:
                raise AssertionError(f"Origin {origin:%Y-%m}: the fit window does not extend the previous one by one month.")
        previous_fit = fit_positions

        empty_in_fit = [feature for feature in APPROVED_FEATURES if fit[feature].isna().all()]
        not_history = [feature for feature in empty_in_fit if not feature.startswith(HISTORY_FEATURE_PREFIX)]
        if not_history:
            raise AssertionError(f"Origin {origin:%Y-%m}: non-history predictors are entirely missing: {not_history}.")
        unobserved_at_origin = [feature for feature in empty_in_fit if validation[feature].isna().all()]
        if unobserved_at_origin:
            raise AssertionError(f"Origin {origin:%Y-%m}: {unobserved_at_origin} are missing in both windows.")

        y_validation = validation[TARGET_COLUMN].to_numpy(dtype=float)
        naive_mae = mean_absolute_error(y_validation, validation["DEMAND_LAG_1M"].fillna(0.0).to_numpy(dtype=float))
        reference = NOTEBOOK_14_NAIVE_MAE_BY_ORIGIN.get(f"{origin:%Y-%m}")
        if reference is not None and abs(naive_mae - reference) > NAIVE_REFERENCE_TOLERANCE:
            raise AssertionError(
                f"Origin {origin:%Y-%m}: naive MAE {naive_mae:.4f} does not reproduce notebook 15's {reference:.3f}."
            )
        rows.append({
            "origin": f"{origin:%Y-%m}",
            "fit_window": f"{fit[TEMPORAL_COLUMN].min():%Y-%m} to {fit[TEMPORAL_COLUMN].max():%Y-%m}",
            "fit_months": fit[TEMPORAL_COLUMN].nunique(),
            "fit_rows": len(fit),
            "validation_rows": len(validation),
            "validation_zero_share": float(np.mean(y_validation == 0)),
            "predictors_empty_in_fit": ", ".join(empty_in_fit) or "none",
            "naive_validation_mae": naive_mae,
        })
    return pd.DataFrame(rows)


rolling_origin_audit = audit_rolling_origins(training_data, TEMPORAL_FOLDS, validation_periods)
print(rolling_origin_audit.round(4).to_string(index=False))
print(f"\nMean naive validation MAE across origins: {rolling_origin_audit['naive_validation_mae'].mean():.3f} "
      "(notebook 15: 30.980)")
BURN_IN_EMPTY_PREDICTORS = sorted({
    feature
    for listed in rolling_origin_audit["predictors_empty_in_fit"] if listed != "none"
    for feature in listed.split(", ")
})
print("Predictors entirely missing in at least one fit window (history burn-in only):", BURN_IN_EMPTY_PREDICTORS or "none")

 origin         fit_window  fit_months  fit_rows  validation_rows  validation_zero_share predictors_empty_in_fit  naive_validation_mae
2025-07 2025-02 to 2025-06           5      4238             1086                 0.4052           DEMAND_LAG_6M               37.4705
2025-08 2025-02 to 2025-07           6      5324             1154                 0.4463                    none               28.7426
2025-09 2025-02 to 2025-08           7      6478             1238                 0.5291                    none               26.7278

Mean naive validation MAE across origins: 30.980 (notebook 14: 30.980)
Predictors entirely missing in at least one fit window (history burn-in only): ['DEMAND_LAG_6M']


## 4. MAE-first scoring on the business scale

MAE is the refit and ranking metric. Predictions are clipped at zero before every metric because the target is non-negative. RMSE, R², MAE on `log1p`, and predicted-to-observed volume ratio are retained as secondary validation diagnostics. The Linear Regression pipeline fits `log1p(y)` and returns `expm1` predictions through `TransformedTargetRegressor`, so search scoring receives predictions on the original billed-volume scale.


In [6]:
def clip_predictions(y_pred):
    return np.clip(np.asarray(y_pred, dtype=float), 0, None)


def clipped_mae(y_true, y_pred):
    return mean_absolute_error(y_true, clip_predictions(y_pred))


def clipped_rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, clip_predictions(y_pred))


def clipped_r2(y_true, y_pred):
    return r2_score(y_true, clip_predictions(y_pred))


def clipped_log_mae(y_true, y_pred):
    return mean_absolute_error(np.log1p(np.asarray(y_true, dtype=float)), np.log1p(clip_predictions(y_pred)))


def volume_ratio(y_true, y_pred):
    denominator = np.asarray(y_true, dtype=float).sum()
    return np.nan if denominator == 0 else clip_predictions(y_pred).sum() / denominator


SCORING = {
    "mae": make_scorer(clipped_mae, greater_is_better=False),
    "rmse": make_scorer(clipped_rmse, greater_is_better=False),
    "r2": make_scorer(clipped_r2),
    "mae_log1p": make_scorer(clipped_log_mae, greater_is_better=False),
    "volume_ratio": make_scorer(volume_ratio),
}


## 5. Model specifications and search budgets

The specifications reuse, rather than replace, the completed candidate work:

- **Linear Regression** uses notebook 16 Section 14.8's `linear_regression_v2`: the six historical-demand features are transformed with `log1p`, the `PORTE_ORDINAL` sentinel is split into a cleaned value and a flag, `MONTH` remains raw, median imputation is fit inside the pipeline, and the target uses `log1p`/`expm1`. The estimator is plain `LinearRegression`, so it has no artificial parameter grid. It receives one fixed-specification expanding-window CV evaluation. Ridge is not substituted: notebook 16 tested it separately and retained plain Linear Regression for v2.
- **Linear Regression -- regularized grid (new, Section 5.1)** searches `Ridge`, `Lasso` and `ElasticNet` together -- `model__regressor` itself is a grid parameter -- over `alpha`, `l1_ratio` (ElasticNet only) and `impute__strategy`, on the same v2 feature specification and the same raw-scale MAE, so it is comparable to the row above and to the tree and forest. This closes the recorded plan of scoring Ridge alone (Section 4.4.6, item 4) with a genuine Ridge candidate rather than an ElasticNet approximation of one.
- **Decision Tree** uses notebook 15's complete 7 × 4 × 3 grid. `GridSearchCV` is appropriate because the finite, registered 84-candidate grid is itself part of that analysis and is affordable at 252 validation fits. Missing values remain native to the tree; no invented imputation or scaling is added.
- **Random Forest** uses notebook 17's four named configurations. `GridSearchCV` is appropriate because exhaustive evaluation of four curated candidates requires only 12 validation fits and exactly preserves the completed tuning work. A broader randomized distribution would silently replace that work. The shared executor nevertheless supports `RandomizedSearchCV` for a future explicitly approved distribution and requires an `n_iter` budget and seed.


**Caveat — `DEMAND_LAG_6M` in the earliest fold:** no unit has six months of prior history yet in the earliest validation fold (2025-07), so that fold's training slice has `DEMAND_LAG_6M` entirely `NaN`. `SimpleImputer` drops the column for that fit and scikit-learn raises a `UserWarning` (`Skipping features without any observed values`). This is the lag burn-in described in notebook 13's contract, not a data-quality bug; it is limited to that one fold and does not affect the other folds or the final refit. The rolling-origin audit (Section 3.1) lists every predictor that is entirely missing in a fit window and fails if one is not a history feature, and the full search (Section 9) silences only this specific warning, so any other warning stays visible.

In [7]:
HISTORICAL_DEMAND_FEATURES = [
    "DEMAND_LAG_1M", "DEMAND_LAG_3M", "DEMAND_LAG_6M",
    "DEMAND_MEAN_3M", "DEMAND_MEAN_6M", "DEMAND_STD_3M",
]
LINEAR_DERIVED_FEATURES = HISTORICAL_DEMAND_FEATURES + [
    "PORTE_ORDINAL_CLEAN", "PORTE_SEM_BENEFICIARIOS", "MONTH",
]
required_linear_inputs = set(HISTORICAL_DEMAND_FEATURES + ["PORTE_ORDINAL", "MONTH"])
if not required_linear_inputs <= set(APPROVED_FEATURES):
    raise ValueError("The persisted feature contract no longer supports the Linear Regression v2 specification.")


def build_linear_v2_features(frame):
    if not isinstance(frame, pd.DataFrame):
        frame = pd.DataFrame(frame, columns=APPROVED_FEATURES)
    transformed = frame.copy()
    transformed[HISTORICAL_DEMAND_FEATURES] = np.log1p(transformed[HISTORICAL_DEMAND_FEATURES])
    transformed["PORTE_ORDINAL_CLEAN"] = transformed["PORTE_ORDINAL"].where(
        transformed["PORTE_ORDINAL"].ne(-1), 0
    )
    transformed["PORTE_SEM_BENEFICIARIOS"] = transformed["PORTE_ORDINAL"].eq(-1).astype(int)
    return transformed[LINEAR_DERIVED_FEATURES]


linear_pipeline = Pipeline([
    ("feature_builder", FunctionTransformer(build_linear_v2_features, validate=False)),
    ("impute", SimpleImputer(strategy="median")),
    ("model", TransformedTargetRegressor(
        regressor=LinearRegression(),
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )),
])

tree_pipeline = Pipeline([
    ("model", DecisionTreeRegressor(random_state=RANDOM_SEED)),
])

forest_pipeline = Pipeline([
    ("model", RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)),
])

TREE_PARAMETER_GRID = {
    "model__max_depth": [3, 4, 5, 6, 8, 10, None],
    "model__min_samples_leaf": [20, 50, 100, 200],
    # Descending order preserves notebook 15's tie-break toward more pruning when MAE ties exactly.
    "model__ccp_alpha": [1e-2, 1e-3, 0.0],
}

FOREST_PARAMETER_GRID = [
    {"model__n_estimators": [150], "model__max_depth": [6], "model__min_samples_leaf": [100]},
    {"model__n_estimators": [300], "model__max_depth": [12], "model__min_samples_leaf": [20]},
    {"model__n_estimators": [600], "model__max_depth": [None], "model__min_samples_leaf": [5]},
    {"model__n_estimators": [600], "model__max_depth": [20], "model__min_samples_leaf": [50]},
]


# The default regressor below is a placeholder for the harness's own pre-flight parameter
# validation (Section 6): every candidate in REGULARIZED_LINEAR_PARAMETER_GRID sets
# model__regressor explicitly (Ridge, Lasso or ElasticNet), so this placeholder is never
# actually fit. It has to be ElasticNet specifically -- not Ridge or Lasso -- because
# validate_model_spec checks every grid key (including model__regressor__l1_ratio, used only
# by the ElasticNet block) against get_params(deep=True) on this untouched pipeline, and only
# ElasticNet exposes both alpha and l1_ratio for that check to find.
regularized_linear_pipeline = Pipeline([
    ("feature_builder", FunctionTransformer(build_linear_v2_features, validate=False)),
    ("impute", SimpleImputer(strategy="median")),
    ("model", TransformedTargetRegressor(
        regressor=ElasticNet(max_iter=50_000, random_state=RANDOM_SEED),
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )),
])

# alpha is a fixed six-point log-spaced grid (0.001 to 100), set independently of any
# held-out look: it is not the sub-range notebook 16's own test-set alpha sweep (Section 13)
# happened to favour, since Section 4.4.1 rules out choices driven by test-set behaviour.
# l1_ratio for the ElasticNet block stays strictly between 0 and 1 (0.2-0.95) because the
# two edges are already the genuine Ridge and Lasso blocks below -- model__regressor itself
# is a grid parameter, so this searches over which estimator to use, not an approximation of
# Ridge/Lasso through ElasticNet's l1_ratio extremes. impute__strategy is searched alongside
# them because Section 5.1 below treats preprocessing choices as part of "was it optimized,"
# not a side note.
REGULARIZED_LINEAR_ALPHAS = [0.001, 0.01, 0.1, 1, 10, 100]
REGULARIZED_LINEAR_IMPUTE_STRATEGIES = ["median", "mean"]
ELASTICNET_L1_RATIOS = [0.2, 0.5, 0.8, 0.95]

REGULARIZED_LINEAR_PARAMETER_GRID = [
    {
        "model__regressor": [Ridge(random_state=RANDOM_SEED)],
        "model__regressor__alpha": REGULARIZED_LINEAR_ALPHAS,
        "impute__strategy": REGULARIZED_LINEAR_IMPUTE_STRATEGIES,
    },
    {
        "model__regressor": [Lasso(max_iter=50_000, random_state=RANDOM_SEED)],
        "model__regressor__alpha": REGULARIZED_LINEAR_ALPHAS,
        "impute__strategy": REGULARIZED_LINEAR_IMPUTE_STRATEGIES,
    },
    {
        "model__regressor": [ElasticNet(max_iter=50_000, random_state=RANDOM_SEED)],
        "model__regressor__alpha": REGULARIZED_LINEAR_ALPHAS,
        "model__regressor__l1_ratio": ELASTICNET_L1_RATIOS,
        "impute__strategy": REGULARIZED_LINEAR_IMPUTE_STRATEGIES,
    },
]


@dataclass(frozen=True)
class ModelSpec:
    name: str
    estimator: Any
    input_features: tuple
    search_strategy: str
    parameter_space: Any
    budget_candidates: int
    random_seed: Any
    source_notebook: str
    strategy_reason: str
    fixed_configuration: Any = None
    n_iter: Any = None


MODEL_SPECS = {
    "linear_regression": ModelSpec(
        name="Linear Regression (v2 fixed specification)",
        estimator=linear_pipeline,
        input_features=tuple(APPROVED_FEATURES),
        search_strategy="fixed_cv",
        parameter_space=None,
        budget_candidates=1,
        random_seed=None,
        source_notebook="16_linear_regression.ipynb, Section 14.8",
        strategy_reason="Plain LinearRegression has no justified tunable hyperparameters; evaluate the completed v2 specification once.",
        fixed_configuration={
            "estimator": "LinearRegression",
            "feature_specification": "linear_regression_v2",
            "target_transform": "log1p/expm1",
        },
    ),
    "linear_regression_regularized": ModelSpec(
        name="Linear Regression -- regularized grid (Ridge/Lasso/ElasticNet)",
        estimator=regularized_linear_pipeline,
        input_features=tuple(APPROVED_FEATURES),
        search_strategy="grid",
        parameter_space=REGULARIZED_LINEAR_PARAMETER_GRID,
        budget_candidates=72,
        random_seed=RANDOM_SEED,
        source_notebook="18_model_comparison.ipynb, Section 5.1 (new)",
        strategy_reason=(
            "Ridge, Lasso and ElasticNet blocks together are 72 candidates (12 + 12 + 48), "
            "smaller than the tree's 84, so the grid is exhausted rather than sampled -- a "
            "RandomizedSearchCV budget was the option's stated fallback and was not needed "
            "once the grid was sized this way. Ridge and Lasso are searched as literal "
            "estimators, not approximated through ElasticNet's l1_ratio extremes, so this "
            "closes the Ridge re-score the recorded plan asked for (Section 4.4.6, item 4) "
            "rather than only bracketing it. Uses the same v2 feature specification and the "
            "same raw-scale MAE the other candidates are ranked on, unlike notebook 16's "
            "Ridge grid, which was never compared on either (Section 4.4.2)."
        ),
    ),
    "decision_tree": ModelSpec(
        name="Decision Tree",
        estimator=tree_pipeline,
        input_features=tuple(APPROVED_FEATURES),
        search_strategy="grid",
        parameter_space=TREE_PARAMETER_GRID,
        budget_candidates=84,
        random_seed=RANDOM_SEED,
        source_notebook="15_decision_tree.ipynb, Section 4",
        strategy_reason="Exhaust the finite registered 7 x 4 x 3 grid used by the completed tree analysis.",
    ),
    "random_forest": ModelSpec(
        name="Random Forest",
        estimator=forest_pipeline,
        input_features=tuple(APPROVED_FEATURES),
        search_strategy="grid",
        parameter_space=FOREST_PARAMETER_GRID,
        budget_candidates=4,
        random_seed=RANDOM_SEED,
        source_notebook="17_random_forest.ipynb, Section 4",
        strategy_reason="Exhaust the four curated configurations from the completed forest analysis; sampling would not save a meaningful budget.",
    ),
}


### 5.1 A broader regularized-linear grid (Ridge, Lasso, ElasticNet)

The recorded plan (Section 4.4.6, item 4) was narrower than this: "score Ridge on the notebook 18 harness," reusing notebook 16's alpha-only grid on the `v2` specification. This section replaces that plan with a broader one: `Ridge`, `Lasso` and `ElasticNet`, searched together as one `GridSearchCV` over `model__regressor` itself, plus `alpha`, `l1_ratio` (ElasticNet only) and the imputation strategy. `model__regressor` is a literal grid parameter, so Ridge and Lasso are genuine candidates in this search, not `l1_ratio` extremes of a single ElasticNet estimator standing in for them.

**This is the broader option, taken with its costs stated rather than discovered later:**

- **Two feature-selection mechanisms can disagree.** `v2`'s six historical-demand features already passed the three-of-four step rule of notebook 16 (Sections 4.3.7.9-4.3.7.10). If the grid below selects Lasso or a high-`l1_ratio` ElasticNet, that estimator's own penalty can zero out one of the two correlated rolling means (`DEMAND_MEAN_3M`, `DEMAND_MEAN_6M`) that rule chose to keep -- a second, competing feature-selection mechanism acting on features the first one already approved. Section 9.3 checks the selected configuration for exactly this and states which mechanism this project treats as deciding when they disagree.
- **More candidates on the same three noisy folds.** Section 4.4.6 already flags `v2`'s validation lead as resting on three folds with a fold-to-fold standard deviation (6.38) more than twice the forest's. Widening the search to 72 candidates on those same three folds raises the chance that the winner is whichever configuration got lucky on them, not the one that generalizes best. Section 9.2 runs the same nested cross-validation notebook 15 ran for the tree -- inner windows choose a configuration, an outer month that took no part in that choice scores it -- to measure how much of this grid's minimum is that kind of optimism.
- **Budget.** Ridge (12 candidates) + Lasso (12) + ElasticNet (48) = 72; kept exhaustive (`GridSearchCV`) rather than sampled, since that is smaller than the tree's already-affordable 84.

`alpha` is a fixed six-point log-spaced grid (0.001-100), set independently of any held-out look: it is not the sub-range notebook 16's own test-set alpha sweep (Section 13) happened to favour, since Section 4.4.1 rules out choices driven by test-set behaviour. `l1_ratio` for the ElasticNet block stays strictly between 0 and 1 (0.2-0.95) precisely because the edges are already the genuine Ridge and Lasso blocks above.


## 6. Specification and nested-parameter validation

Pipeline parameters must use valid nested names such as `model__max_depth`. This validation runs before any fitting, checks the declared budget against the actual grid size, and requires randomized searches to declare a positive `n_iter` no larger than their budget.


In [8]:
def parameter_keys(parameter_space):
    if parameter_space is None:
        return set()
    grids = parameter_space if isinstance(parameter_space, list) else [parameter_space]
    return {key for grid in grids for key in grid}


def candidate_count(spec: ModelSpec):
    if spec.search_strategy == "fixed_cv":
        return 1
    if spec.search_strategy == "grid":
        return len(ParameterGrid(spec.parameter_space))
    if spec.search_strategy == "randomized":
        if not isinstance(spec.n_iter, int) or spec.n_iter <= 0:
            raise ValueError(f"{spec.name}: randomized search requires a positive integer n_iter.")
        return spec.n_iter
    raise ValueError(f"{spec.name}: unsupported search strategy {spec.search_strategy!r}.")


def validate_model_spec(spec: ModelSpec):
    unknown_features = sorted(set(spec.input_features) - set(APPROVED_FEATURES))
    if unknown_features:
        raise ValueError(f"{spec.name}: features are outside the persisted contract: {unknown_features}.")
    valid_parameters = set(spec.estimator.get_params(deep=True))
    invalid_parameters = sorted(parameter_keys(spec.parameter_space) - valid_parameters)
    if invalid_parameters:
        raise ValueError(f"{spec.name}: invalid nested estimator parameters: {invalid_parameters}.")
    actual_candidates = candidate_count(spec)
    if actual_candidates != spec.budget_candidates:
        raise ValueError(
            f"{spec.name}: declared budget is {spec.budget_candidates} candidates, "
            f"but the specification creates {actual_candidates}."
        )
    if spec.search_strategy == "randomized" and spec.random_seed is None:
        raise ValueError(f"{spec.name}: randomized search requires a seed.")
    return {
        "model": spec.name,
        "strategy": spec.search_strategy,
        "candidates": actual_candidates,
        "validation_fits": actual_candidates * len(TEMPORAL_FOLDS),
        "total_fits_including_refit": actual_candidates * len(TEMPORAL_FOLDS) + 1,
        "seed": spec.random_seed,
        "source": spec.source_notebook,
        "reason": spec.strategy_reason,
    }


search_plan = pd.DataFrame([validate_model_spec(spec) for spec in MODEL_SPECS.values()])
search_plan


,model,strategy,candidates,validation_fits,total_fits_including_refit,seed,source,reason
0,Linear Regression (v2 fixed specification),fixed_cv,1,3,4,NaN,"15_linear_regression_demand_model.ipynb, Secti...",Plain LinearRegression has no justified tunabl...
1,Linear Regression -- regularized grid (Ridge/L...,grid,72,216,217,42.0,"17_supervised_candidate_search_workflow.ipynb,...","Ridge, Lasso and ElasticNet blocks together ar..."
2,Decision Tree,grid,84,252,253,42.0,"14_decision_tree_candidate.ipynb, Section 4",Exhaust the finite registered 7 x 4 x 3 grid u...
3,Random Forest,grid,4,12,13,42.0,"16_random_forest_demand_model.ipynb, Section 4",Exhaust the four curated configurations from t...


## 7. Shared execution and common result format

`cv_results_to_common_table` converts every `GridSearchCV` or `RandomizedSearchCV` result into the same candidate-level schema. The fixed Linear Regression evaluator produces that schema directly without pretending an empty grid is a hyperparameter search. `selected_configuration` means the best configuration **within that model's validation run**; it does not mean the project selected that model. Validation and holdout metrics remain separate tables.


In [9]:
def serialize_parameters(parameters):
    return json.dumps(parameters, sort_keys=True, default=str)


def cv_results_to_common_table(search, spec: ModelSpec):
    raw = pd.DataFrame(search.cv_results_)
    n_candidates = len(raw)
    return pd.DataFrame({
        "model_name": spec.name,
        "search_strategy": spec.search_strategy,
        "candidate_parameters": raw["params"].map(serialize_parameters),
        "candidate_rank": raw[f"rank_test_{PRIMARY_METRIC}"].astype(int),
        "mean_validation_mae": -raw[f"mean_test_{PRIMARY_METRIC}"],
        "validation_mae_std": raw[f"std_test_{PRIMARY_METRIC}"],
        "mean_validation_rmse": -raw["mean_test_rmse"],
        "mean_validation_r2": raw["mean_test_r2"],
        "mean_validation_mae_log1p": -raw["mean_test_mae_log1p"],
        "mean_validation_volume_ratio": raw["mean_test_volume_ratio"],
        "mean_fit_time_seconds": raw["mean_fit_time"],
        "selected_configuration": np.arange(n_candidates) == search.best_index_,
        "number_evaluated_candidates": n_candidates,
        "estimated_validation_fits": n_candidates * len(TEMPORAL_FOLDS),
        "estimated_total_fits_including_refit": n_candidates * len(TEMPORAL_FOLDS) + 1,
        "search_seed": spec.random_seed,
        "search_status": "complete",
    })


def fixed_cv_to_common_table(cv_output, spec: ModelSpec):
    return pd.DataFrame([{
        "model_name": spec.name,
        "search_strategy": spec.search_strategy,
        "candidate_parameters": serialize_parameters(spec.fixed_configuration),
        "candidate_rank": 1,
        "mean_validation_mae": -np.mean(cv_output["test_mae"]),
        "validation_mae_std": np.std(-cv_output["test_mae"]),
        "mean_validation_rmse": -np.mean(cv_output["test_rmse"]),
        "mean_validation_r2": np.mean(cv_output["test_r2"]),
        "mean_validation_mae_log1p": -np.mean(cv_output["test_mae_log1p"]),
        "mean_validation_volume_ratio": np.mean(cv_output["test_volume_ratio"]),
        "mean_fit_time_seconds": np.mean(cv_output["fit_time"]),
        "selected_configuration": True,
        "number_evaluated_candidates": 1,
        "estimated_validation_fits": len(TEMPORAL_FOLDS),
        "estimated_total_fits_including_refit": len(TEMPORAL_FOLDS) + 1,
        "search_seed": spec.random_seed,
        "search_status": "complete",
    }])


def instantiate_search(spec: ModelSpec):
    common = dict(
        estimator=clone(spec.estimator),
        scoring=SCORING,
        refit=PRIMARY_METRIC,
        cv=TEMPORAL_FOLDS,
        n_jobs=SEARCH_N_JOBS,
        error_score="raise",
        return_train_score=False,
    )
    if spec.search_strategy == "grid":
        return GridSearchCV(param_grid=spec.parameter_space, **common)
    if spec.search_strategy == "randomized":
        return RandomizedSearchCV(
            param_distributions=spec.parameter_space,
            n_iter=spec.n_iter,
            random_state=spec.random_seed,
            **common,
        )
    raise ValueError(f"{spec.name}: {spec.search_strategy!r} does not instantiate a search object.")


def run_model(spec: ModelSpec):
    validate_model_spec(spec)
    X_training = training_data[list(spec.input_features)]
    y_training = training_data[TARGET_COLUMN]
    if spec.search_strategy == "fixed_cv":
        cv_output = cross_validate(
            clone(spec.estimator), X_training, y_training,
            scoring=SCORING, cv=TEMPORAL_FOLDS, n_jobs=SEARCH_N_JOBS, error_score="raise",
        )
        fitted_estimator = clone(spec.estimator).fit(X_training, y_training)
        return {
            "spec": spec, "search": None, "estimator": fitted_estimator,
            "results": fixed_cv_to_common_table(cv_output, spec),
        }

    search = instantiate_search(spec)
    search.fit(X_training, y_training)
    return {
        "spec": spec, "search": search, "estimator": search.best_estimator_,
        "results": cv_results_to_common_table(search, spec),
    }


def run_models(model_names, specifications=MODEL_SPECS):
    unknown_names = sorted(set(model_names) - set(specifications))
    if unknown_names:
        raise KeyError(f"Unknown model names: {unknown_names}. Available: {list(specifications)}")
    outcomes = {name: run_model(specifications[name]) for name in model_names}
    result_table = pd.concat([outcome["results"] for outcome in outcomes.values()], ignore_index=True)
    return outcomes, result_table


def validation_leaderboard(common_results):
    selected_rows = common_results.loc[common_results["selected_configuration"]].copy()
    return selected_rows.sort_values("mean_validation_mae", kind="stable").reset_index(drop=True)


## 8. Optional smoke test — non-final

The smoke specifications keep the real pipelines, data contract, temporal folds, and scoring, but reduce the tree and forest spaces to one cheap candidate each. To run the smoke test manually, temporarily set `RUN_SMOKE_TEST = True`, then restore it to `False`. Never use this table for model selection, never publish its scores, and never persist its fitted estimators as final artifacts.

In [10]:
RUN_SMOKE_TEST = False

SMOKE_SPECS = {
    "linear_regression": MODEL_SPECS["linear_regression"],
    "decision_tree": replace(
        MODEL_SPECS["decision_tree"],
        parameter_space={
            "model__max_depth": [3],
            "model__min_samples_leaf": [100],
            "model__ccp_alpha": [0.0],
        },
        budget_candidates=1,
        strategy_reason="NON-FINAL smoke configuration only.",
    ),
    "random_forest": replace(
        MODEL_SPECS["random_forest"],
        parameter_space=[{
            "model__n_estimators": [10],
            "model__max_depth": [4],
            "model__min_samples_leaf": [100],
        }],
        budget_candidates=1,
        strategy_reason="NON-FINAL smoke configuration only.",
    ),
}

if RUN_SMOKE_TEST:
    print("NON-FINAL SMOKE TEST: reduced spaces; scores are not model-selection evidence.")
    smoke_outcomes, smoke_results = run_models(list(SMOKE_SPECS), SMOKE_SPECS)
    display(smoke_results)
else:
    print("Smoke test skipped (RUN_SMOKE_TEST=False).")

Smoke test skipped (RUN_SMOKE_TEST=False).


## 9. Run one full search or all full searches

Set `FULL_SEARCH_MODELS = ["decision_tree"]` for one candidate, or `FULL_SEARCH_MODELS = list(MODEL_SPECS)` for every registered specification. The Linear Regression entry performs fixed-specification CV rather than a fake search. Review `full_results` for candidate-level detail and `full_validation_leaderboard` for the best configuration per model, ranked only by validation MAE. The harness defaults to `FULL_SEARCH_MODELS = []` (no full run) when idle; this checked-in run sets it to `list(MODEL_SPECS)` below to exercise every registered specification.


In [11]:
FULL_SEARCH_MODELS = list(MODEL_SPECS)

if FULL_SEARCH_MODELS:
    with warnings.catch_warnings():
        # Only the expected burn-in warning is silenced (Section 3.1): SimpleImputer drops a
        # predictor that is entirely missing in a fit window. Every other warning stays visible.
        warnings.filterwarnings(
            "ignore", message="Skipping features without any observed values", category=UserWarning,
        )
        full_outcomes, full_results = run_models(FULL_SEARCH_MODELS)
    full_validation_leaderboard = validation_leaderboard(full_results)
    display(full_results.sort_values(["model_name", "candidate_rank"]))
    display(full_validation_leaderboard)
else:
    full_outcomes, full_results, full_validation_leaderboard = {}, pd.DataFrame(), pd.DataFrame()
    print("No full search executed. Configure FULL_SEARCH_MODELS when the team is ready.")

,model_name,search_strategy,candidate_parameters,candidate_rank,mean_validation_mae,validation_mae_std,mean_validation_rmse,mean_validation_r2,mean_validation_mae_log1p,mean_validation_volume_ratio,mean_fit_time_seconds,selected_configuration,number_evaluated_candidates,estimated_validation_fits,estimated_total_fits_including_refit,search_seed,search_status
85,Decision Tree,grid,"{""model__ccp_alpha"": 0.01, ""model__max_depth"":...",1,31.332727,4.442816,152.911487,0.353491,0.692654,1.333291,0.004122,True,84,252,253,42,complete
113,Decision Tree,grid,"{""model__ccp_alpha"": 0.001, ""model__max_depth""...",1,31.332727,4.442816,152.911487,0.353491,0.692654,1.333291,0.004345,False,84,252,253,42,complete
141,Decision Tree,grid,"{""model__ccp_alpha"": 0.0, ""model__max_depth"": ...",1,31.332727,4.442816,152.911487,0.353491,0.692654,1.333291,0.004566,False,84,252,253,42,complete
149,Decision Tree,grid,"{""model__ccp_alpha"": 0.0, ""model__max_depth"": ...",4,31.855712,4.878133,153.500837,0.349686,0.616975,1.333991,0.005885,False,84,252,253,42,complete
121,Decision Tree,grid,"{""model__ccp_alpha"": 0.001, ""model__max_depth""...",5,31.864573,4.878245,153.500843,0.349686,0.624747,1.334191,0.005772,False,84,252,253,42,complete
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,Linear Regression -- regularized grid (Ridge/L...,grid,"{""impute__strategy"": ""mean"", ""model__regressor...",55,50.127403,15.361612,218.048711,-0.045871,1.634449,0.102454,0.002223,False,72,216,217,42,complete
159,Random Forest,grid,"{""model__max_depth"": null, ""model__min_samples...",1,31.011184,2.433362,143.778174,0.273499,0.609683,1.389197,0.491063,True,4,12,13,42,complete
158,Random Forest,grid,"{""model__max_depth"": 12, ""model__min_samples_l...",2,31.258393,3.257644,153.727720,0.365301,0.615383,1.376378,0.203761,False,4,12,13,42,complete
160,Random Forest,grid,"{""model__max_depth"": 20, ""model__min_samples_l...",3,32.936333,2.720058,166.211537,0.317011,0.620810,1.383752,0.377810,False,4,12,13,42,complete


,model_name,search_strategy,candidate_parameters,candidate_rank,mean_validation_mae,validation_mae_std,mean_validation_rmse,mean_validation_r2,mean_validation_mae_log1p,mean_validation_volume_ratio,mean_fit_time_seconds,selected_configuration,number_evaluated_candidates,estimated_validation_fits,estimated_total_fits_including_refit,search_seed,search_status
0,Linear Regression -- regularized grid (Ridge/L...,grid,"{""impute__strategy"": ""mean"", ""model__regressor...",1,24.167522,6.192935,126.529547,0.587438,0.531580,0.904356,0.003071,True,72,216,217,42,complete
1,Linear Regression (v2 fixed specification),fixed_cv,"{""estimator"": ""LinearRegression"", ""feature_spe...",1,24.322655,6.379932,127.206640,0.587021,0.529211,0.899459,0.005217,True,1,3,4,None,complete
2,Random Forest,grid,"{""model__max_depth"": null, ""model__min_samples...",1,31.011184,2.433362,143.778174,0.273499,0.609683,1.389197,0.491063,True,4,12,13,42,complete
3,Decision Tree,grid,"{""model__ccp_alpha"": 0.01, ""model__max_depth"":...",1,31.332727,4.442816,152.911487,0.353491,0.692654,1.333291,0.004122,True,84,252,253,42,complete


## 9.1 Consistency with the candidate notebooks

Notebooks 15 and 17 run the same searches as custom loops over the same folds, so `GridSearchCV` here must search the same grids and select the same configurations with the same validation MAE. The reference values below are copied from those notebooks' committed outputs and serve only this check; they select nothing. `max_depth = None` is compared as missing on both sides.

| Candidate | Reference | Where it is printed |
|---|---|---|
| Decision Tree | 84-candidate grid; `max_depth = 6`, `min_samples_leaf = 20`, `ccp_alpha = 0.01`; validation MAE 31.33272744607466 | notebook 15, cells 21 and 36; notebook 17, cell 18 |
| Random Forest | the four `RF_CONFIGS`; `rf_deep_large` (600 trees, `max_depth = None`, `min_samples_leaf = 5`); validation MAE 31.0112 | notebook 17, cells 12 and 57 |

In [12]:
from itertools import product

CANDIDATE_NOTEBOOK_REFERENCE = {
    "decision_tree": {
        "source": "notebook 15 (cells 21, 36); notebook 17 (cell 18)",
        "grid": {"max_depth": [3, 4, 5, 6, 8, 10, None], "min_samples_leaf": [20, 50, 100, 200],
                 "ccp_alpha": [0.0, 1e-3, 1e-2]},
        "selected": {"max_depth": 6, "min_samples_leaf": 20, "ccp_alpha": 0.01},
        "validation_mae": 31.33272744607466,
        "tolerance": 1e-9,
    },
    "random_forest": {
        "source": "notebook 17 (cells 12, 57)",
        "grid": [
            {"n_estimators": 150, "max_depth": 6, "min_samples_leaf": 100},
            {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 20},
            {"n_estimators": 600, "max_depth": None, "min_samples_leaf": 5},
            {"n_estimators": 600, "max_depth": 20, "min_samples_leaf": 50},
        ],
        "selected": {"n_estimators": 600, "max_depth": None, "min_samples_leaf": 5},
        "validation_mae": 31.0112,
        "tolerance": 5e-5,  # notebook 17 prints the forest's validation MAE to four decimals
    },
}


def strip_prefix(parameters):
    return {key.removeprefix("model__"): value for key, value in parameters.items()}


def same_value(left, right):
    if left is None or right is None:
        return left is None and right is None
    return float(left) == float(right)


def same_configuration(left, right):
    return left.keys() == right.keys() and all(same_value(left[key], right[key]) for key in left)


def reference_configurations(grid):
    if isinstance(grid, list):
        return grid
    return [dict(zip(grid, values)) for values in product(*grid.values())]


consistency_rows = []
for model_key, reference in CANDIDATE_NOTEBOOK_REFERENCE.items():
    if model_key not in full_outcomes:
        print(f"{model_key}: not in FULL_SEARCH_MODELS, consistency check skipped.")
        continue
    spec = MODEL_SPECS[model_key]
    searched = [strip_prefix(parameters) for parameters in ParameterGrid(spec.parameter_space)]
    expected = reference_configurations(reference["grid"])
    same_space = len(searched) == len(expected) and all(
        any(same_configuration(candidate, other) for other in searched) for candidate in expected
    )
    selected = strip_prefix(full_outcomes[model_key]["search"].best_params_)
    selected_row = full_results.loc[
        full_results["model_name"].eq(spec.name) & full_results["selected_configuration"]
    ].iloc[0]
    mae_gap = abs(selected_row["mean_validation_mae"] - reference["validation_mae"])
    consistency_rows.append({
        "candidate": spec.name,
        "reference": reference["source"],
        "same_search_space": same_space,
        "same_selected_configuration": same_configuration(selected, reference["selected"]),
        "selected_here": json.dumps(selected, sort_keys=True),
        "validation_mae_here": selected_row["mean_validation_mae"],
        "validation_mae_reference": reference["validation_mae"],
        "abs_difference": mae_gap,
        "within_tolerance": mae_gap <= reference["tolerance"],
    })

configuration_consistency = pd.DataFrame(consistency_rows)
display(configuration_consistency)
check_columns = ["same_search_space", "same_selected_configuration", "within_tolerance"]
failed_checks = configuration_consistency.loc[
    ~configuration_consistency[check_columns].all(axis=1), "candidate"
].tolist()
if failed_checks:
    raise AssertionError(f"GridSearchCV disagrees with the candidate notebooks for: {failed_checks}.")
print("GridSearchCV reproduces notebooks 15 and 17: same search spaces, same selected configurations, same validation MAE.")

,candidate,reference,same_search_space,same_selected_configuration,selected_here,validation_mae_here,validation_mae_reference,abs_difference,within_tolerance
0,Decision Tree,"notebook 14 (cells 21, 36); notebook 16 (cell 18)",True,True,"{""ccp_alpha"": 0.01, ""max_depth"": 6, ""min_sampl...",31.332727,31.332727,0.000000,True
1,Random Forest,"notebook 16 (cells 12, 57)",True,True,"{""max_depth"": null, ""min_samples_leaf"": 5, ""n_...",31.011184,31.011200,0.000016,True


GridSearchCV reproduces notebooks 14 and 16: same search spaces, same selected configurations, same validation MAE.


### 9.2 Nested cross-validation for the regularized-linear grid

Section 5.1 flagged that 72 candidates on three folds risk selecting whichever configuration got lucky on them. This section measures that risk exactly as notebook 15 did for the tree (Section 4.3.8.3 there): for each of the three validation origins, the **two training months immediately before it** choose a configuration by lowest mean MAE over those two inner folds, and the origin itself -- which took no part in that choice -- scores only the configuration those inner folds picked. The mean of those three genuinely out-of-sample scores is the nested estimate; the gap to the non-nested minimum of Section 9 is how optimistic that minimum is.

This is a diagnostic on the search, not a re-selection: it changes no configuration and feeds nothing back into `full_results` or the validation leaderboard.


In [13]:
REGULARIZED_KEY = "linear_regression_regularized"


def _fit_predict_mae(params, fit_rows, val_rows):
    features = list(MODEL_SPECS[REGULARIZED_KEY].input_features)
    X_fit = training_data.iloc[fit_rows][features]
    y_fit = training_data.iloc[fit_rows][TARGET_COLUMN]
    X_val = training_data.iloc[val_rows][features]
    y_val = training_data.iloc[val_rows][TARGET_COLUMN]
    candidate = clone(regularized_linear_pipeline).set_params(**params)
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message="Skipping features without any observed values", category=UserWarning,
        )
        candidate.fit(X_fit, y_fit)
        predictions = candidate.predict(X_val)
    return clipped_mae(y_val.to_numpy(dtype=float), predictions)


def _mean_inner_mae(params, inner_folds):
    return float(np.mean([_fit_predict_mae(params, fit_rows, val_rows) for fit_rows, val_rows in inner_folds]))


def _describe_params(params):
    # Display-only: model__regressor holds an estimator instance, not a JSON-serializable
    # value, so this is never used to reconstruct params (that keeps the real dict, with the
    # real estimator object, in memory the whole time -- see the loop below).
    described = {}
    for key, value in params.items():
        if key == "model__regressor":
            described["regressor"] = type(value).__name__
        else:
            described[key.replace("model__regressor__", "").replace("impute__", "impute_")] = value
    return json.dumps(described, sort_keys=True, default=str)


if REGULARIZED_KEY in full_outcomes:
    training_months = pd.DatetimeIndex(sorted(training_data[TEMPORAL_COLUMN].unique()))
    candidate_params = list(ParameterGrid(REGULARIZED_LINEAR_PARAMETER_GRID))

    nested_rows = []
    for (fit_positions, validation_positions), outer_month in zip(TEMPORAL_FOLDS, validation_periods):
        position = training_months.get_loc(outer_month)
        inner_months = training_months[position - 2:position]
        inner_folds = [
            (
                np.flatnonzero(training_data[TEMPORAL_COLUMN].lt(month)),
                np.flatnonzero(training_data[TEMPORAL_COLUMN].eq(month)),
            )
            for month in inner_months
        ]
        # A list of (score, params) pairs, not a dict keyed by params: model__regressor holds
        # an estimator instance, which is not a stable/meaningful dict key.
        inner_scores = [(_mean_inner_mae(params, inner_folds), params) for params in candidate_params]
        inner_mae, best_params = min(inner_scores, key=lambda item: item[0])
        outer_mae = _fit_predict_mae(best_params, fit_positions, validation_positions)
        nested_rows.append({
            "outer_month": f"{outer_month:%Y-%m}",
            "chosen_on": ",".join(f"{month:%b}" for month in inner_months),
            "chosen_params": _describe_params(best_params),
            "inner_MAE": inner_mae,
            "outer_MAE": outer_mae,
        })

    nested_regularized = pd.DataFrame(nested_rows)
    nested_regularized_estimate = nested_regularized["outer_MAE"].mean()
    non_nested_minimum = full_validation_leaderboard.loc[
        full_validation_leaderboard["model_name"].eq(MODEL_SPECS[REGULARIZED_KEY].name),
        "mean_validation_mae",
    ].iloc[0]
    # Positive means the non-nested minimum underestimated out-of-sample error (optimistic);
    # negative would mean the nested estimate scored better than the reported minimum.
    optimism = nested_regularized_estimate - non_nested_minimum

    print(f"{len(nested_regularized)} outer months x {len(candidate_params)} candidates x 2 inner folds = "
          f"{len(nested_regularized) * len(candidate_params) * 2} fits.")
    display(nested_regularized.round(3))
    print(f"\nNested estimate (mean over the outer months): {nested_regularized_estimate:.3f}")
    print(f"Non-nested minimum reported in Section 9:      {non_nested_minimum:.3f}")
    print(f"Optimism of the non-nested number (nested minus non-nested): {optimism:+.3f}")
else:
    nested_regularized = pd.DataFrame()
    print(f"{REGULARIZED_KEY} not in FULL_SEARCH_MODELS; nested CV skipped.")


3 outer months x 72 candidates x 2 inner folds = 432 fits.


,outer_month,chosen_on,chosen_params,inner_MAE,outer_MAE
0,2025-07,"May,Jun","{""alpha"": 0.001, ""impute_strategy"": ""mean"", ""r...",40.885,32.411
1,2025-08,"Jun,Jul","{""alpha"": 0.001, ""impute_strategy"": ""mean"", ""r...",35.095,21.751
2,2025-09,"Jul,Aug","{""alpha"": 0.01, ""impute_strategy"": ""mean"", ""l1...",27.059,18.416



Nested estimate (mean over the outer months): 24.193
Non-nested minimum reported in Section 9:      24.168
Optimism of the non-nested number (nested minus non-nested): +0.025


### 9.3 Does the selected configuration overlap with the three-of-four rule?

Section 5.1's first caveat is checked here rather than left as a hypothetical: if the grid selects Lasso or a high-`l1_ratio` ElasticNet, and that estimator's penalty pushed the coefficient on `DEMAND_MEAN_3M` or `DEMAND_MEAN_6M` to (near) zero, it would have made the same call the three-of-four rule already made when it kept both features in `v2` (Sections 4.3.7.9-4.3.7.10). Ridge's L2 penalty does not induce sparse feature selection, so its coefficients are reported without interpreting a small value as an L1-driven feature drop. This does not happen for every configuration on the grid -- only, potentially, for the one `GridSearchCV` actually selected.


In [14]:
if REGULARIZED_KEY in full_outcomes:
    selected_estimator = full_outcomes[REGULARIZED_KEY]["estimator"]
    best_params = full_outcomes[REGULARIZED_KEY]["search"].best_params_
    selected_regressor_type = type(best_params["model__regressor"]).__name__
    fitted_regressor = selected_estimator.named_steps["model"].regressor_
    rolling_mean_features = ["DEMAND_MEAN_3M", "DEMAND_MEAN_6M"]
    coefficients = pd.Series(fitted_regressor.coef_, index=LINEAR_DERIVED_FEATURES)
    near_zero = coefficients.loc[rolling_mean_features].abs() < 1e-8

    print(f"Selected regressor: {selected_regressor_type}")
    print(coefficients.loc[rolling_mean_features].round(6))
    if selected_regressor_type == "Ridge":
        print("\nRidge selected: L2 regularization does not induce sparse feature selection. "
              "The coefficients above are not interpreted as an L1-driven feature drop.")
    elif near_zero.any():
        print(f"\nOverlap confirmed: {selected_regressor_type}'s penalty zeroed "
              f"{list(near_zero.index[near_zero])} at the selected configuration -- a second "
              "feature-selection mechanism acting on features the three-of-four rule already "
              "approved. This project treats the regularization grid's own selection as "
              "informative about that specific configuration, not as a re-run of the "
              "three-of-four rule; v2's feature set (Sections 4.3.7.9-4.3.7.10) remains the "
              "approved specification unless this regularized candidate is separately promoted.")
    else:
        print(f"\nNo overlap: both rolling-mean coefficients stay non-zero at the selected "
              f"{selected_regressor_type} configuration, so its penalty did not re-decide what "
              "the three-of-four rule already settled.")
else:
    print(f"{REGULARIZED_KEY} not in FULL_SEARCH_MODELS; selection-overlap check skipped.")


Selected regressor: ElasticNet
DEMAND_MEAN_3M    0.290600
DEMAND_MEAN_6M    0.237493
dtype: float64

No overlap: both rolling-mean coefficients stay non-zero at the selected ElasticNet configuration, so its penalty did not re-decide what the three-of-four rule already settled.


## 10. Holdout gate

Holdout evaluation is allowed only after a complete documented validation run and an explicit team decision. It is a final estimate, not another tuning signal. The function below reads only the already refitted estimator for the declared model and returns a separate one-row holdout table. It never appends holdout metrics to the validation ranking.


In [15]:
def evaluate_selected_on_holdout(outcomes, selected_model_name):
    if selected_model_name not in outcomes:
        raise ValueError(
            "The selected model must have a completed full validation run in this session before holdout evaluation."
        )
    outcome = outcomes[selected_model_name]
    spec = outcome["spec"]
    y_true = holdout_data[TARGET_COLUMN].to_numpy(dtype=float)
    y_pred = clip_predictions(outcome["estimator"].predict(holdout_data[list(spec.input_features)]))
    naive_pred = holdout_data["DEMAND_LAG_1M"].fillna(0.0).to_numpy(dtype=float)
    naive_mae = mean_absolute_error(y_true, naive_pred)
    model_mae = mean_absolute_error(y_true, y_pred)
    return pd.DataFrame([{
        "model_name": spec.name,
        "evaluation_partition": "final_holdout",
        "holdout_mae": model_mae,
        "holdout_rmse": root_mean_squared_error(y_true, y_pred),
        "holdout_r2": r2_score(y_true, y_pred),
        "holdout_mae_log1p": mean_absolute_error(np.log1p(y_true), np.log1p(y_pred)),
        "holdout_volume_ratio": volume_ratio(y_true, y_pred),
        "holdout_skill_vs_naive": 1 - model_mae / naive_mae,
        "selection_note": "Evaluated only after validation-based model and configuration selection.",
    }])


ALLOW_HOLDOUT_EVALUATION = False
FINAL_SELECTED_MODEL = None

if ALLOW_HOLDOUT_EVALUATION:
    if FINAL_SELECTED_MODEL is None:
        raise ValueError("Set FINAL_SELECTED_MODEL only after the team completes validation-based selection.")
    final_holdout_results = evaluate_selected_on_holdout(full_outcomes, FINAL_SELECTED_MODEL)
    display(final_holdout_results)
else:
    final_holdout_results = pd.DataFrame()
    print("Holdout remains untouched. This notebook has not selected a final model.")


Holdout remains untouched. This notebook has not selected a final model.


## Output interpretation and limits

- **Structural outputs**: resolved contract, discovered periods, fold boundaries, the rolling-origin audit (Section 3.1), model specifications, parameter validation, and search budgets. These may be inspected without running full searches.
- **Consistency outputs**: `configuration_consistency` (Section 9.1), which checks that `GridSearchCV` reproduces the custom searches of notebooks 15 and 17 and fails the run on any disagreement.
- **Regularized-linear diagnostics**: `nested_regularized` (Section 9.2) and the selection-overlap check (Section 9.3) are diagnostics about the search itself, not additional validation evidence; they select nothing and feed nothing back into `full_results` or `full_validation_leaderboard`.
- **Validation outputs**: `full_results` and `full_validation_leaderboard`. These may select configurations and compare candidates using MAE, but they are not final performance estimates.
- **Final outputs**: `final_holdout_results`, produced only through the explicit holdout gate after selection. They must remain separate from validation ranking.
- **Smoke outputs**: reduced, non-final integration checks only. They are not reusable scores, selected configurations, or model artifacts.

This harness deliberately does not duplicate the candidate notebooks' diagnostics, robustness analyses, feature-importance work, or final-model discussion. If the persisted matrix or feature contract changes, resolve that incompatibility in the producing workstream before editing this notebook's assertions.
